In [ ]:
# Pun Khattgar - Assignment 1

## Building a POS Tagger using myPOS Version 3.0 and CRF

In [ ]:
!python --version

In [3]:
cd "/home/psyche-khattgar/POS Tagging"

/home/psyche-khattgar/POS Tagging


In [4]:
pwd

'/home/psyche-khattgar/POS Tagging'

In [5]:
ls

mypos-ver.3.0.shuf.nopipe.txt


In [10]:
!ls -lah "/home/psyche-khattgar/POS Tagging"

total 9.2M
drwxrwxr-x  2 psyche-khattgar psyche-khattgar 4.0K Aug 17 15:42 .
drwxr-x--- 31 psyche-khattgar psyche-khattgar 4.0K Aug 17 15:42 ..
-rw-rw-r--  1 psyche-khattgar psyche-khattgar 9.2M Aug  6 20:58 mypos-ver.3.0.shuf.nopipe.txt


In [11]:
!head -n 10 "/home/psyche-khattgar/POS Tagging/mypos-ver.3.0.shuf.nopipe.txt"

၁၉၆၂/num ခုနှစ်/n ခန့်မှန်း/v သန်းခေါင်စာရင်း/n အရ/ppm လူဦးရေ/n ၁၁၅၉၃၁/num ယောက်/part ရှိ/v သည်/ppm ။/punc
လူ/n တိုင်း/part တွင်/ppm သင့်မြတ်/v လျော်ကန်/v စွာ/part ကန့်သတ်/v ထား/part သည့်/part အလုပ်/n လုပ်/v ချိန်/n အပြင်/conj ၊/punc လစာ/n နှင့်တကွ/conj အခါ/n ကာလ/n အားလျော်စွာ/ppm သတ်မှတ်/v ထား/part သည့်/part အလုပ်/n အားလပ်ရက်/n များ/part ပါဝင်/v သည့်/part အနားယူခွင့်/n နှင့်/conj အားလပ်ခွင့်/n ခံစားပိုင်ခွင့်/n ရှိ/v သည်/ppm ။/punc
ဤ/adj နည်း/n ကို/ppm စစ်ယူ/v သော/part နည်း/n ဟု/part ခေါ်/v သည်/ppm ။/punc
စာပြန်ပွဲ/n ဆို/v တာ/part က/ppm အာဂုံဆောင်/v အလွတ်ကျက်/v ထား/part တဲ့/part ပိဋကတ်သုံးပုံ/n စာပေ/n တွေ/part ကို/ppm စာစစ်/v သံဃာတော်ကြီး/n တွေ/part ရဲ့/ppm ရှေ့/n မှာ/ppm အလွတ်/adv ပြန်/v ပြီး/part ရွတ်ပြ/v ရ/part တာ/part ပေါ့/part ။/punc
ဒီ/pron မှာ/ppm ကျွန်တော့်/pron သက်သေခံကတ်/n ပါ/part ။/punc
၂ဝ/num ရာစု/n မြန်မာ့/n သမိုင်း/n သန်းဝင်းလှိုင်/n ၊/punc ၂ဝဝ၉/num ခု/part ၊/punc မေ/n လ/n ၊/punc ကံကော်ဝတ်ရည်/n စာပေ/n ။/punc
ကျွန်တော်/pron မျက်မှန်/n တစ်/tn လက်/part လုပ်/v ချင်/part ပါ/p

In [12]:
def convert_mypos(input_file, output_file):
    with open(input_file, encoding="utf-8") as f:
        lines = f.readlines()

    with open(output_file, "w", encoding="utf-8") as out:
        for line in lines:
            tokens = line.strip().split()

            for token in tokens:
                if "/" not in token:
                    continue

                word, tag = token.rsplit("/", 1)
                out.write(f"{word} {tag}\n")

            out.write("\n")

In [14]:
convert_mypos(
    "/home/psyche-khattgar/POS Tagging/mypos-ver.3.0.shuf.nopipe.txt",
    "/home/psyche-khattgar/POS Tagging/mypos.train.txt"
)

In [15]:
!wget -O "/home/psyche-khattgar/POS Tagging/otest.1k.nopipe.txt" \
"https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/master/corpus-ver-3.0/otest.1k.nopipe.txt"

--2026-08-17 16:05:01--  https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/master/corpus-ver-3.0/otest.1k.nopipe.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-08-17 16:05:02 ERROR 404: Not Found.



In [16]:
convert_mypos(
    "/home/psyche-khattgar/POS Tagging/otest.1k.nopipe.txt",
    "/home/psyche-khattgar/POS Tagging/mypos.test.txt"
)

In [17]:
%%writefile mypos_features.py

import crfutils

fields = 'w y'
separator = ' '

templates = (
    (('w', -2), ),
    (('w', -1), ),
    (('w', 0), ),
    (('w', 1), ),
    (('w', 2), ),
    (('w', -1), ('w', 0)),
    (('w', 0), ('w', 1)),
)

def feature_extractor(X):
    crfutils.apply_templates(X, templates)

    if X:
        X[0]['F'].append('__BOS__')
        X[-1]['F'].append('__EOS__')

if __name__ == '__main__':
    crfutils.main(
        feature_extractor,
        fields=fields,
        sep=separator
    )

Writing mypos_features.py


In [18]:
!cat "/home/psyche-khattgar/POS Tagging/mypos.train.txt" | \
python mypos_features.py \
> "/home/psyche-khattgar/POS Tagging/train.crfsuite.txt"

Traceback (most recent call last):
  File "/home/psyche-khattgar/POS Tagging/mypos_features.py", line 2, in <module>
    import crfutils
ModuleNotFoundError: No module named 'crfutils'
cat: write error: Broken pipe


In [19]:
!cat "/home/psyche-khattgar/POS Tagging/mypos.test.txt" | \
python "/home/psyche-khattgar/POS Tagging/mypos_features.py" \
> "/home/psyche-khattgar/POS Tagging/test.crfsuite.txt"

Traceback (most recent call last):
  File "/home/psyche-khattgar/POS Tagging/mypos_features.py", line 2, in <module>
    import crfutils
ModuleNotFoundError: No module named 'crfutils'


In [2]:
!ls -lh "/home/psyche-khattgar/POS Tagging/train.crfsuite.txt"

-rw-rw-r-- 1 psyche-khattgar psyche-khattgar 0 Aug 17 16:07 '/home/psyche-khattgar/POS Tagging/train.crfsuite.txt'


In [3]:
!ls -lah "/home/psyche-khattgar/POS Tagging/"

total 19M
drwxrwxr-x  2 psyche-khattgar psyche-khattgar 4.0K Aug 17 16:08 .
drwxr-x--- 32 psyche-khattgar psyche-khattgar 4.0K Aug 17 16:17 ..
-rw-rw-r--  1 psyche-khattgar psyche-khattgar  490 Aug 17 16:06 mypos_features.py
-rw-rw-r--  1 psyche-khattgar psyche-khattgar    0 Aug 17 16:06 mypos.test.txt
-rw-rw-r--  1 psyche-khattgar psyche-khattgar 9.2M Aug 17 15:57 mypos.train.txt
-rw-rw-r--  1 psyche-khattgar psyche-khattgar 9.2M Aug  6 20:58 mypos-ver.3.0.shuf.nopipe.txt
-rw-rw-r--  1 psyche-khattgar psyche-khattgar    0 Aug 17 16:05 otest.1k.nopipe.txt
-rw-rw-r--  1 psyche-khattgar psyche-khattgar    0 Aug 17 16:08 test.crfsuite.txt
-rw-rw-r--  1 psyche-khattgar psyche-khattgar    0 Aug 17 16:07 train.crfsuite.txt


In [4]:
!python "/home/psyche-khattgar/POS Tagging/mypos_features.py" \
< "/home/psyche-khattgar/POS Tagging/mypos.train.txt" \
> "/home/psyche-khattgar/POS Tagging/train.crfsuite.txt"

Traceback (most recent call last):
  File "/home/psyche-khattgar/POS Tagging/mypos_features.py", line 2, in <module>
    import crfutils
ModuleNotFoundError: No module named 'crfutils'


In [7]:
import sys
print(sys.executable)

/home/psyche-khattgar/anaconda3/bin/python


In [8]:
!{sys.executable} -m pip install crfutils

ERROR: Could not find a version that satisfies the requirement crfutils (from versions: none)
ERROR: No matching distribution found for crfutils


In [9]:
import crfutils

ModuleNotFoundError: No module named 'crfutils'

In [10]:
!ls -lh "/home/psyche-khattgar/POS Tagging/crfutils.py"

ls: cannot access '/home/psyche-khattgar/POS Tagging/crfutils.py': No such file or directory


In [11]:
!head -n 20 "/home/psyche-khattgar/POS Tagging/crfutils.txt"

head: cannot open '/home/psyche-khattgar/POS Tagging/crfutils.txt' for reading: No such file or directory


In [ ]:
!find "/home/psyche-khattgar" -iname "crfutils*" 2>/dev/null

/home/psyche-khattgar/crfsuite/example/crfutils.py
/home/psyche-khattgar/POS Tagging/crfutils.py


In [1]:
!wget -O "/home/psyche-khattgar/POS Tagging/crfutils.py" \
https://raw.githubusercontent.com/chokkan/crfsuite/master/example/crfutils.py

--2026-08-17 16:49:31--  https://raw.githubusercontent.com/chokkan/crfsuite/master/example/crfutils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6193 (6.0K) [text/plain]
Saving to: ‘/home/psyche-khattgar/POS Tagging/crfutils.py’

/home/psyche-khattg 100%[===================>]   6.05K  --.-KB/s    in 0.001s  

2026-08-17 16:49:32 (8.97 MB/s) - ‘/home/psyche-khattgar/POS Tagging/crfutils.py’ saved [6193/6193]



In [1]:
import sys
sys.path.insert(0, "/home/psyche-khattgar/POS Tagging")

import crfutils
print("crfutils OK")

crfutils OK


In [4]:
!python "/home/psyche-khattgar/POS Tagging/mypos_features.py" \
< "/home/psyche-khattgar/POS Tagging/mypos.train.txt" \
> "/home/psyche-khattgar/POS Tagging/train.crfsuite.txt"

In [3]:
print("hello")

hello


In [5]:
!ls -lh "/home/psyche-khattgar/POS Tagging/train.crfsuite.txt"

-rw-rw-r-- 1 psyche-khattgar psyche-khattgar 87M Aug 17 18:07 '/home/psyche-khattgar/POS Tagging/train.crfsuite.txt'


In [6]:
!time crfsuite learn \
-m "/home/psyche-khattgar/POS Tagging/pos_tag.model" \
"/home/psyche-khattgar/POS Tagging/train.crfsuite.txt"

CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

Start time of the training: 2026-08-17T11:39:09Z

Reading the data set(s)
[1] /home/psyche-khattgar/POS Tagging/train.crfsuite.txt
0....1....2....3....4....5....6....7....8....9....10
Number of instances: 43197
Seconds required: 8.532

Statistics the data set(s)
Number of data sets (groups): 1
Number of instances: 43196
Number of items: 564517
Number of attributes: 452474
Number of labels: 15

Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 0
0....1....2....3....4....5....6....7....8....9....10
Number of features: 571865
Seconds required: 3.009

L-BFGS optimization
c1: 0.000000
c2: 1.000000
num_memories: 6
max_iterations: 2147483647
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

***** Iteration #1 *****
Loss: 963962.209493
Feature norm: 5.000000
Error norm: 77493.052921
Active features: 571865
Line search trials: 2

In [7]:
!ls -lh "/home/psyche-khattgar/POS Tagging/pos_tag.model"

-rw-rw-r-- 1 psyche-khattgar psyche-khattgar 47M Aug 17 18:15 '/home/psyche-khattgar/POS Tagging/pos_tag.model'


In [8]:
!wget -O "/home/psyche-khattgar/POS Tagging/otest.1k.nopipe.txt" \
"https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus/otest.1k.nopipe.txt"

--2026-08-17 18:17:58--  https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus/otest.1k.nopipe.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 229758 (224K) [text/plain]
Saving to: ‘/home/psyche-khattgar/POS Tagging/otest.1k.nopipe.txt’

/home/psyche-khattg 100%[===================>] 224.37K   720KB/s    in 0.3s    

2026-08-17 18:18:00 (720 KB/s) - ‘/home/psyche-khattgar/POS Tagging/otest.1k.nopipe.txt’ saved [229758/229758]



In [9]:
!ls -lh "/home/psyche-khattgar/POS Tagging/otest.1k.nopipe.txt"

-rw-rw-r-- 1 psyche-khattgar psyche-khattgar 225K Aug 17 18:18 '/home/psyche-khattgar/POS Tagging/otest.1k.nopipe.txt'


In [12]:
def convert_mypos(input_file, output_file):
    with open(input_file, encoding="utf-8") as f:
        lines = f.readlines()

    with open(output_file, "w", encoding="utf-8") as out:
        for line in lines:
            tokens = line.strip().split()

            for token in tokens:
                if "/" not in token:
                    continue

                word, tag = token.rsplit("/", 1)
                out.write(f"{word} {tag}\n")

            out.write("\n")

In [14]:
convert_mypos(
    "/home/psyche-khattgar/POS Tagging/otest.1k.nopipe.txt",
    "/home/psyche-khattgar/POS Tagging/mypos.test.txt"
)

In [15]:
!ls -lh "/home/psyche-khattgar/POS Tagging/mypos.test.txt"

-rw-rw-r-- 1 psyche-khattgar psyche-khattgar 226K Aug 17 18:20 '/home/psyche-khattgar/POS Tagging/mypos.test.txt'


In [16]:
!python "/home/psyche-khattgar/POS Tagging/mypos_features.py" \
< "/home/psyche-khattgar/POS Tagging/mypos.test.txt" \
> "/home/psyche-khattgar/POS Tagging/test.crfsuite.txt"

In [17]:
!ls -lh "/home/psyche-khattgar/POS Tagging/test.crfsuite.txt"

-rw-rw-r-- 1 psyche-khattgar psyche-khattgar 2.1M Aug 17 18:20 '/home/psyche-khattgar/POS Tagging/test.crfsuite.txt'


In [18]:
!head -n 10 "/home/psyche-khattgar/POS Tagging/test.crfsuite.txt"

tn	w[0]=တစ်	w[1]=ကိုက်	w[2]=ကို	w[0]|w[1]=တစ်|ကိုက်	__BOS__
n	w[-1]=တစ်	w[0]=ကိုက်	w[1]=ကို	w[2]=ဝမ်	w[-1]|w[0]=တစ်|ကိုက်	w[0]|w[1]=ကိုက်|ကို
ppm	w[-2]=တစ်	w[-1]=ကိုက်	w[0]=ကို	w[1]=ဝမ်	w[2]=ခုနှစ်ထောင်	w[-1]|w[0]=ကိုက်|ကို	w[0]|w[1]=ကို|ဝမ်
n	w[-2]=ကိုက်	w[-1]=ကို	w[0]=ဝမ်	w[1]=ခုနှစ်ထောင်	w[2]=ပါ	w[-1]|w[0]=ကို|ဝမ်	w[0]|w[1]=ဝမ်|ခုနှစ်ထောင်
tn	w[-2]=ကို	w[-1]=ဝမ်	w[0]=ခုနှစ်ထောင်	w[1]=ပါ	w[2]=။	w[-1]|w[0]=ဝမ်|ခုနှစ်ထောင်	w[0]|w[1]=ခုနှစ်ထောင်|ပါ
part	w[-2]=ဝမ်	w[-1]=ခုနှစ်ထောင်	w[0]=ပါ	w[1]=။	w[-1]|w[0]=ခုနှစ်ထောင်|ပါ	w[0]|w[1]=ပါ|။
punc	w[-2]=ခုနှစ်ထောင်	w[-1]=ပါ	w[0]=။	w[-1]|w[0]=ပါ|။	__EOS__

n	w[0]=မနှစ်	w[1]=က	w[2]=သူ	w[0]|w[1]=မနှစ်|က	__BOS__
ppm	w[-1]=မနှစ်	w[0]=က	w[1]=သူ	w[2]=ကျွန်မ	w[-1]|w[0]=မနှစ်|က	w[0]|w[1]=က|သူ


In [19]:
!time crfsuite tag \
-m "/home/psyche-khattgar/POS Tagging/pos_tag.model" \
"/home/psyche-khattgar/POS Tagging/test.crfsuite.txt" \
> "/home/psyche-khattgar/POS Tagging/test.tag.out"


real	0m0.194s
user	0m0.115s
sys	0m0.068s


In [20]:
!ls -lh "/home/psyche-khattgar/POS Tagging/test.tag.out"

-rw-rw-r-- 1 psyche-khattgar psyche-khattgar 49K Aug 17 18:21 '/home/psyche-khattgar/POS Tagging/test.tag.out'


In [21]:
!head -n 50 "/home/psyche-khattgar/POS Tagging/test.tag.out"

tn
n
ppm
n
num
part
punc

n
ppm
pron
pron
ppm
v
part
ppm
punc

pron
n
v
v
part
punc

n
v
part
v
ppm
part
punc

n
adv
v
part
punc
tn
tn
n
part
v
part
conj
v
part
ppm
part
punc


In [22]:
!time crfsuite tag \
-r \
-m "/home/psyche-khattgar/POS Tagging/pos_tag.model" \
"/home/psyche-khattgar/POS Tagging/test.crfsuite.txt" \
> "/home/psyche-khattgar/POS Tagging/ref_hyp.out.txt"


real	0m0.234s
user	0m0.120s
sys	0m0.064s


In [23]:
!head -n 50 "/home/psyche-khattgar/POS Tagging/ref_hyp.out.txt"

tn	tn
n	n
ppm	ppm
n	n
tn	num
part	part
punc	punc

n	n
ppm	ppm
pron	pron
pron	pron
ppm	ppm
v	v
part	part
ppm	ppm
punc	punc

pron	pron
n	n
v	v
v	v
part	part
punc	punc

n	n
v	v
part	part
v	v
ppm	ppm
part	part
punc	punc

n	n
adv	adv
v	v
part	part
punc	punc
tn	tn
tn	tn
n	n
part	part
v	v
part	part
conj	conj
v	v
part	part
ppm	ppm
part	part
punc	punc


In [26]:
!time crfsuite tag \
-qt \
-m "/home/psyche-khattgar/POS Tagging/pos_tag.model" \
"/home/psyche-khattgar/POS Tagging/test.crfsuite.txt"

Performance by label (#match, #model, #ref) (precision, recall, F1):
    num: (146, 147, 155) (0.9932, 0.9419, 0.9669)
    n: (2963, 3102, 3000) (0.9552, 0.9877, 0.9712)
    v: (1948, 2007, 2010) (0.9706, 0.9692, 0.9699)
    ppm: (2035, 2063, 2060) (0.9864, 0.9879, 0.9871)
    part: (3134, 3185, 3189) (0.9840, 0.9828, 0.9834)
    punc: (1270, 1270, 1270) (1.0000, 1.0000, 1.0000)
    conj: (394, 415, 411) (0.9494, 0.9586, 0.9540)
    adj: (324, 345, 366) (0.9391, 0.8852, 0.9114)
    adv: (233, 246, 262) (0.9472, 0.8893, 0.9173)
    pron: (459, 469, 476) (0.9787, 0.9643, 0.9714)
    tn: (138, 138, 142) (1.0000, 0.9718, 0.9857)
    fw: (46, 46, 87) (1.0000, 0.5287, 0.6917)
    int: (23, 23, 25) (1.0000, 0.9200, 0.9583)
    sb: (3, 3, 3) (1.0000, 1.0000, 1.0000)
    abb: (9, 9, 12) (1.0000, 0.7500, 0.8571)
Macro-average precision, recall, F1: (0.980251, 0.915828, 0.941700)
Item accuracy: 13125 / 13468 (0.9745)
Instance accuracy: 747 / 1000 (0.7470)
Elapsed time: 0.111509 [sec] (8967.9 [ins

In [28]:
!crfsuite dump "/home/psyche-khattgar/POS Tagging/pos_tag.model" | head -n 30

FILEHEADER = {
  magic: lCRF
  size: 48409000
  type: FOMC
  version: 100
  num_features: 0
  num_labels: 15
  num_attrs: 452474
  off_features: 0x30
  off_labels: 0xAE8530
  off_attrs: 0xAE8F25
  off_labelrefs: 0x28885DC
  off_attrrefs: 0x28889A0
}

LABELS = {
      0: num
      1: n
      2: v
      3: ppm
      4: part
      5: punc
      6: conj
      7: adj
      8: adv
      9: pron
     10: tn
     11: fw
     12: int
     13: sb
